# Réseau de neurones simple avec PyTorch Lightning

Dans ces travaux pratiques, nous construisons avec PyTorch Lightning des réseaux de neurones qui reconnaissent les chiffres manuscrits de MNIST. Lightning écrit la boucle d'entraînement à notre place : le modèle reste un `nn.Module` PyTorch, et un `Trainer` l'entraîne (slides « PyTorch Lightning », « Définir un `LightningModule` » et « Entraîner avec le `Trainer` »).

1. une régression logistique multinomiale, le réseau sans couche cachée, avec deux façons de donner les classes à la perte ;
2. un perceptron multicouche régularisé ;
3. une recherche d'hyper-paramètres avec Optuna ;
4. le suivi de l'entraînement avec TensorBoard, pour comparer deux optimiseurs.

Il n'est pas nécessaire d'utiliser une carte graphique pour ce TP : tout est rapide sur CPU. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import lightning
import matplotlib.pyplot as plt
import optuna
import torch
import torchmetrics
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Récupération des données

Comme dans le TP précédent, torchvision fournit le jeu MNIST.

In [ ]:
donnees_entrainement = torchvision.datasets.MNIST("donnees", train=True, download=True)
donnees_test = torchvision.datasets.MNIST("donnees", train=False, download=True)
X_train, y_train = donnees_entrainement.data, donnees_entrainement.targets
X_test, y_test = donnees_test.data, donnees_test.targets

## Regardons les données

Étudiez comment sont stockées les données :

- la forme et le type des quatre tenseurs ;
- la valeur des premiers labels et l'intervalle des pixels ;
- quelques images, avec leur label ;
- le nombre d'exemples de chaque classe : les classes sont-elles équilibrées ?

**Documentation utile**

- [`torch.bincount`](https://docs.pytorch.org/docs/stable/generated/torch.bincount.html)
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()` et `X_train.max()`
- `torch.bincount(y_train)` : le nombre d'occurrences de chaque entier, de 0 à 9
- `plt.subplots(1, 10)` renvoie une figure et 10 axes ; `ax.imshow(image, cmap="gray_r")` affiche une image 28 × 28 en niveaux de gris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(X_train.dtype, X_train.min(), X_train.max(), y_train.dtype, y_train[:10])
print(torch.bincount(...))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(...)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
print(f"Format de X_train : {tuple(X_train.shape)}, de X_test : {tuple(X_test.shape)}")
print(f"Format de y_train : {tuple(y_train.shape)}, de y_test : {tuple(y_test.shape)}")
print(f"Type des images : {X_train.dtype}, de {X_train.min()} à {X_train.max()}")
print(f"Type des labels : {y_train.dtype}, premiers : {y_train[:10].tolist()}")
print(f"Exemples par classe : {torch.bincount(y_train).tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(int(y_train[i]))
  ax.axis("off")
plt.show()

- 60 000 images d'entraînement et 10 000 de test, de 28 × 28 pixels.
- Les images sont des entiers non signés sur 8 bits (`torch.uint8`), de 0 (fond) à 255 (encre) ; les labels, des entiers `torch.int64`, de 0 à 9.
- De 5 421 à 6 742 exemples par classe : des classes à peu près équilibrées.

## Transformation des données

Nous devons transformer ces données :

1. Aplatir les images : un tenseur de forme `(_, 784)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples. Chaque image devient un vecteur de 784 pixels, l'entrée attendue par une couche `nn.Linear`.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt que $[0, 255]$.
3. Encoder les classes en [*one-hot*](https://fr.wikipedia.org/wiki/Encodage_one-hot) : pour chaque exemple, 10 valeurs, toutes nulles sauf un 1 à l'indice de sa classe. C'est la première des deux façons de donner les classes à l'entropie croisée que nous verrons.

Effectuez les deux premières transformations sur `X_train` et `X_test`. Pour la troisième, gardez `y_train` et `y_test`, et rangez les classes encodées dans `Y_train` et `Y_test`.

**Documentation utile**

- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

In [ ]:
nb_classes = 10
dim_entree = 28 * 28

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.reshape(-1, dim_entree)` : `-1` laisse PyTorch calculer le nombre d'exemples
- `/ 255` divise chaque pixel et donne des `torch.float32`
- `nn.functional.one_hot(y_train, nb_classes)` renvoie un tenseur d'entiers de forme `(60000, 10)` ; `.float()` le convertit en flottants, le type qu'attend l'entropie croisée pour des cibles one-hot. Le label 5 y devient `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = X_train.reshape(...) / ...
X_test = ...
Y_train = nn.functional.one_hot(..., nb_classes).float()
Y_test = ...
```

### Solution

In [ ]:
# Des images « à plat » : l'entrée du réseau est un vecteur de dim_entree pixels,
# dans [0, 1]. -1 laisse PyTorch calculer le nombre d'exemples.
X_train = X_train.reshape(-1, dim_entree) / 255
X_test = X_test.reshape(-1, dim_entree) / 255

# Les classes en vecteurs one-hot, en flottants pour l'entropie croisée
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)
print(f"Label {int(y_train[0])} en one-hot : {Y_train[0].tolist()}")

## Le `LightningModule` du TP

La cellule suivante définit les outils des parties suivantes :

- `ClassifieurMNIST`, le `LightningModule` du TP (slide « Définir un `LightningModule` ») : il enveloppe un modèle PyTorch qui renvoie des *logits*, les scores bruts des 10 classes, et se charge de la perte, de la justesse (*accuracy*, avec torchmetrics) et de l'optimiseur. La perte est l'entropie croisée, qui applique elle-même la softmax (slide « Piège classique : softmax et entropie croisée ») et accepte comme cibles des vecteurs one-hot aussi bien que des indices de classe. L'optimiseur, son learning rate et sa *weight decay* sont des paramètres ;
- `fabrique_chargeurs`, qui sépare des données en entraînement et validation (20 % par défaut) et renvoie leurs `DataLoader` ;
- `evalue`, qui affiche la perte et la justesse d'un modèle entraîné sur l'ensemble de test.

In [ ]:
class ClassifieurMNIST(lightning.LightningModule):
  """Enveloppe Lightning : entropie croisée sur les logits, justesse comme métrique."""

  def __init__(
    self,
    modele: nn.Module,
    optimiseur: type[torch.optim.Optimizer] = torch.optim.SGD,
    taux_apprentissage: float = 0.01,
    decroissance_poids: float = 0.0,
    nb_classes_sortie: int = nb_classes,
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele"])
    self.modele = modele
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, dim_entree)
    # torchmetrics demande une instance de métrique par étape
    self.justesses = nn.ModuleDict(
      {
        f"justesse_{etape}": torchmetrics.Accuracy(
          task="multiclass", num_classes=nb_classes_sortie
        )
        for etape in ("entrainement", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    images, cibles = batch
    logits = self(images)
    perte = nn.functional.cross_entropy(logits, cibles)
    # La métrique, elle, a besoin d'indices de classe
    etiquettes = cibles.argmax(dim=-1) if cibles.dim() > 1 else cibles
    justesse = self.justesses[f"justesse_{etape}"]
    justesse(logits, etiquettes)
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"justesse_{etape}", justesse, on_step=False, on_epoch=True, prog_bar=True)
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimiseur(
      self.parameters(),
      lr=self.hparams.taux_apprentissage,
      weight_decay=self.hparams.decroissance_poids,
    )


def fabrique_chargeurs(
  X: torch.Tensor,
  Y: torch.Tensor,
  taille_lot: int = 128,
  fraction_validation: float = 0.2,
) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation."""
  dataset_entrainement, dataset_val = random_split(
    TensorDataset(X, Y), [1 - fraction_validation, fraction_validation]
  )
  return (
    DataLoader(dataset_entrainement, batch_size=taille_lot, shuffle=True),
    DataLoader(dataset_val, batch_size=taille_lot),
  )


def evalue(
  trainer: lightning.Trainer, classifieur: ClassifieurMNIST, one_hot: bool = False
) -> None:
  """Affiche la perte et la justesse sur l'ensemble de test.

  one_hot : True pour un modèle entraîné sur les cibles one-hot (Y_train).
  """
  chargeur_test = DataLoader(
    TensorDataset(X_test, Y_test if one_hot else y_test), batch_size=128
  )
  score = trainer.test(classifieur, chargeur_test, verbose=False)[0]
  print(
    f"Test : perte {score['perte_test']:.3f}, justesse {score['justesse_test']:.3f}"
  )

## Régression logistique multinomiale

Créez un modèle sans couche cachée, `modele` : un `nn.Sequential` qui ne contient qu'une couche `nn.Linear` de `dim_entree` entrées et `nb_classes` sorties. Chaque sortie est une combinaison linéaire des pixels, un logit, et la softmax de l'entropie croisée en fait 10 probabilités de somme 1 : c'est une régression logistique multinomiale.

Enveloppez-le dans `classifieur`, un `ClassifieurMNIST` avec ses réglages par défaut. Affichez son résumé avec `ModelSummary`, et expliquez son nombre de paramètres.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(nn.Linear(dim_entree, nb_classes))`
- `ClassifieurMNIST(modele)` : SGD, avec un learning rate de 0,01
- `print(ModelSummary(classifieur, max_depth=-1))` : `max_depth=-1` détaille toutes les couches, y compris celles du modèle enveloppé
- Résultat attendu : 7 850 paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = nn.Sequential(nn.Linear(..., ...))
classifieur = ClassifieurMNIST(...)
print(ModelSummary(..., max_depth=-1))
```

### Solution

In [ ]:
modele = nn.Sequential(nn.Linear(dim_entree, nb_classes))
classifieur = ClassifieurMNIST(modele)
print(ModelSummary(classifieur, max_depth=-1))

# Une couche linéaire a un poids par entrée et par sortie, plus un biais par
# sortie : 784 × 10 + 10 = 7 850 paramètres

## Apprentissage

Entraînez `classifieur` avec un `lightning.Trainer` (slide « Entraîner avec le `Trainer` ») :

- des batchs de 128 exemples, avec 20 % des données d'entraînement en validation : `fabrique_chargeurs` sur `X_train` et les cibles one-hot `Y_train` ;
- 10 epochs ;
- un `CSVLogger`, qui enregistre les métriques dans `journaux/simple`, et sans sauvegarde de checkpoints.

Mesurez enfin sa perte et sa justesse sur l'ensemble de test avec `evalue`. Gardez ces chiffres : ils servent de référence aux modèles suivants.

**Documentation utile**

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.CSVLogger.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `fabrique_chargeurs(X_train, Y_train, taille_lot=128)` renvoie les `DataLoader` d'entraînement et de validation
- `lightning.Trainer(max_epochs=10, accelerator="auto", devices=1, logger=CSVLogger("journaux", name="simple"), enable_checkpointing=False)` : `accelerator="auto"` prend le GPU s'il y en a un
- `callbacks=[TQDMProgressBar()]` : la barre de progression de tqdm, plutôt que celle de rich, que Lightning choisit quand rich est installé (Colab) et qui a figé des entraînements
- `trainer.fit(classifieur, chargeur_entrainement, chargeur_val)`
- `evalue(trainer, classifieur, one_hot=True)` : le modèle apprend sur des cibles one-hot
- Résultat attendu : environ 89 % de justesse en validation comme en test ; SGD, avec un learning rate de 0,01, progresse encore à la dixième epoch

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
chargeur_entrainement, chargeur_val = fabrique_chargeurs(..., ..., taille_lot=...)
trainer = lightning.Trainer(
  max_epochs=...,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(..., ..., ...)
evalue(trainer, classifieur, one_hot=...)
```

### Solution

In [ ]:
# Le Trainer exécute la boucle d'entraînement complète du LightningModule
chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, Y_train, taille_lot=128
)

trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="simple"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
evalue(trainer, classifieur, one_hot=True)

## Une perte pour les labels entiers

L'entropie croisée de PyTorch accepte aussi, comme cibles, les labels entiers `y_train` (des indices de classe, de type `torch.int64`) : l'encodage one-hot devient inutile.

Créez `classifieur2`, un `ClassifieurMNIST` qui enveloppe un nouveau modèle identique au précédent, et entraînez-le sur `y_train` avec les mêmes réglages (`CSVLogger` dans `journaux/simple_epars`). Comparez sa perte et sa justesse de test à celles du premier modèle : que change le format des cibles ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un nouveau `nn.Sequential(nn.Linear(dim_entree, nb_classes))` : un modèle aux poids neufs
- `fabrique_chargeurs(X_train, y_train, taille_lot=128)`, puis un nouveau `Trainer`, avec `name="simple_epars"`
- `evalue(trainer2, classifieur2)`, sans `one_hot`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
classifieur2 = ClassifieurMNIST(nn.Sequential(nn.Linear(dim_entree, nb_classes)))
chargeur_entrainement2, chargeur_val2 = fabrique_chargeurs(X_train, ..., taille_lot=128)
trainer2 = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer2.fit(classifieur2, ..., ...)
evalue(trainer2, classifieur2)
```

### Solution

In [ ]:
classifieur2 = ClassifieurMNIST(nn.Sequential(nn.Linear(dim_entree, nb_classes)))
chargeur_entrainement2, chargeur_val2 = fabrique_chargeurs(
  X_train, y_train, taille_lot=128
)

trainer2 = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="simple_epars"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer2.fit(classifieur2, chargeur_entrainement2, chargeur_val2)
evalue(trainer2, classifieur2)

# Les mêmes résultats, aux tirages aléatoires près (initialisation, ordre des
# batchs) : seul le format des cibles change. Les labels entiers évitent de
# stocker 10 valeurs par exemple.

## Perceptron multicouche régularisé

Écrivez une fonction `construit_modele_profond` qui renvoie un perceptron multicouche neuf :

- 4 couches cachées de 20 neurones, avec l'activation ReLU ;
- pour toutes les couches, une initialisation orthogonale des matrices de poids (`nn.init.orthogonal_`) ;
- une couche de sortie de `nb_classes` neurones, sans activation : elle renvoie des logits, comme la régression logistique.

Enveloppez-en un dans `classifieur_profond`, un `ClassifieurMNIST` qui l'entraîne avec l'optimiseur `torch.optim.AdamW`, un learning rate de 10⁻³ et une *weight decay* de 0,01 : la régularisation est portée par l'optimiseur (slide « AdamW et régularisation des poids »). Affichez son résumé, puis entraînez-le sur les labels entiers avec les mêmes réglages que la régression logistique. Combien de points de justesse en test gagne-t-il sur elle ? Pour combien de paramètres de plus ?

Les parties suivantes réutilisent `construit_modele_profond`.

**Documentation utile**

- [`torch.nn.init.orthogonal_`](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.orthogonal_)

In [ ]:
def construit_modele_profond() -> nn.Module: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.init.orthogonal_(couche.weight)` remplace sur place les poids d'une couche `nn.Linear` par une matrice orthogonale, qui conserve la norme du signal d'une couche à l'autre
- Une fonction `cree_dense(nb_entrees, nb_sorties)` qui crée la couche et initialise ses poids évite de répéter ces deux lignes
- `nn.Sequential(cree_dense(dim_entree, 20), nn.ReLU(), ...)` : chaque couche cachée est suivie de son activation
- `ClassifieurMNIST(construit_modele_profond(), optimiseur=torch.optim.AdamW, taux_apprentissage=1e-3, decroissance_poids=0.01)`
- Résultat attendu : 17 170 paramètres, environ 95 % de justesse en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cree_dense(nb_entrees: int, nb_sorties: int) -> nn.Linear:
  couche = nn.Linear(nb_entrees, nb_sorties)
  nn.init.orthogonal_(...)
  return couche


def construit_modele_profond() -> nn.Module:
  return nn.Sequential(
    cree_dense(dim_entree, 20),
    nn.ReLU(),
    ...,
    cree_dense(20, nb_classes),
  )


classifieur_profond = ClassifieurMNIST(
  construit_modele_profond(),
  optimiseur=...,
  taux_apprentissage=...,
  decroissance_poids=...,
)
print(ModelSummary(classifieur_profond, max_depth=-1))
chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, y_train, taille_lot=128
)
trainer_profond = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="profond"),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer_profond.fit(..., ..., ...)
evalue(trainer_profond, classifieur_profond)
```

### Solution

In [ ]:
def cree_dense(nb_entrees: int, nb_sorties: int) -> nn.Linear:
  """Une couche linéaire aux poids initialisés par une matrice orthogonale."""
  couche = nn.Linear(nb_entrees, nb_sorties)
  nn.init.orthogonal_(couche.weight)
  return couche


def construit_modele_profond() -> nn.Module:
  """Un perceptron multicouche neuf : 4 couches cachées de 20 neurones."""
  return nn.Sequential(
    cree_dense(dim_entree, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, nb_classes),
  )


# La régularisation est portée par l'optimiseur : AdamW applique la weight decay
# directement aux poids, alors qu'une pénalité L2 ajoutée à la perte serait
# déformée par les pas adaptatifs d'Adam
classifieur_profond = ClassifieurMNIST(
  construit_modele_profond(),
  optimiseur=torch.optim.AdamW,
  taux_apprentissage=1e-3,
  decroissance_poids=0.01,
)
print(ModelSummary(classifieur_profond, max_depth=-1))

chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, y_train, taille_lot=128
)
trainer_profond = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="profond"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer_profond.fit(classifieur_profond, chargeur_entrainement, chargeur_val)
evalue(trainer_profond, classifieur_profond)

# Environ 6 points de plus que la régression logistique (95 % contre 89 %), pour
# 17 170 paramètres au lieu de 7 850. Une partie du gain vient d'AdamW : la partie
# TensorBoard compare les deux optimiseurs sur ce modèle.

## Recherche d'hyper-paramètres

Combien de neurones dans la couche cachée, quel learning rate ? [Optuna](https://optuna.org/) essaie des combinaisons d'hyper-paramètres et garde la meilleure, notée sur l'ensemble de validation. Son échantillonneur par défaut, TPE, choisit chaque essai d'après les résultats des précédents (slide « Pour aller plus loin : recherche guidée et élagage »).

Écrivez une fonction objectif, `objectif(essai)`, qui :

- tire les hyper-paramètres de l'essai : `unites`, le nombre de neurones cachés, de 32 à 512 par pas de 32 ; `taux_apprentissage`, le learning rate d'Adam, de 10⁻⁴ à 10⁻², en échelle logarithmique ;
- entraîne 3 fois un perceptron à une couche cachée ReLU, de sortie `nb_classes` logits, enveloppé dans un `ClassifieurMNIST` avec l'optimiseur `torch.optim.Adam`, sur les labels entiers : 10 epochs, des batchs de 1 500 exemples, 20 % de validation ;
- renvoie la moyenne des 3 justesses de validation finales, moins bruitée qu'une seule.

Créez ensuite une étude qui maximise cet objectif, lancez-la sur 5 essais, puis affichez le tableau des essais, les meilleurs hyper-paramètres et la meilleure justesse.

**Documentation utile**

- [`optuna.trial.Trial`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html)
- [`optuna.create_study`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html)
- [`optuna.study.Study.optimize`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html#optuna.study.Study.optimize)

In [ ]:
def objectif(essai: optuna.Trial) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `essai.suggest_int("unites", 32, 512, step=32)` et `essai.suggest_float("taux_apprentissage", 1e-4, 1e-2, log=True)` renvoient la valeur de l'essai en cours
- `nn.Sequential(nn.Linear(dim_entree, unites), nn.ReLU(), nn.Linear(unites, nb_classes))`
- `lightning.Trainer(..., logger=False, enable_progress_bar=False, enable_model_summary=False)` : un entraînement silencieux, pour que les 15 entraînements ne remplissent pas la sortie
- `trainer.callback_metrics["justesse_val"]` : la justesse de validation de la dernière epoch, un tenseur
- `optuna.create_study(direction="maximize")` : une étude qui cherche le maximum de l'objectif ; `etude.optimize(objectif, n_trials=5)` la lance
- `etude.trials_dataframe()`, `etude.best_params` et `etude.best_value` : les essais, les meilleurs hyper-paramètres et la meilleure valeur de l'objectif

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def objectif(essai: optuna.Trial) -> float:
  unites = essai.suggest_int("unites", ...)
  taux_apprentissage = essai.suggest_float("taux_apprentissage", ...)
  justesses_val = []
  for _ in range(3):
    modele = nn.Sequential(
      nn.Linear(dim_entree, ...), nn.ReLU(), nn.Linear(..., nb_classes)
    )
    classifieur = ClassifieurMNIST(modele, optimiseur=..., taux_apprentissage=...)
    chargeur_entrainement, chargeur_val = fabrique_chargeurs(
      X_train, y_train, taille_lot=...
    )
    trainer = lightning.Trainer(
      max_epochs=10,
      accelerator="auto",
      devices=1,
      logger=False,
      enable_checkpointing=False,
      enable_progress_bar=False,
      enable_model_summary=False,
    )
    trainer.fit(...)
    justesses_val.append(float(trainer.callback_metrics[...]))
  return ...


etude = optuna.create_study(direction=...)
etude.optimize(objectif, n_trials=...)
print(etude.trials_dataframe())
print(etude.best_params, etude.best_value)
```

### Solution

In [ ]:
NB_EXECUTIONS_PAR_ESSAI = 3


def objectif(essai: optuna.Trial) -> float:
  unites = essai.suggest_int("unites", 32, 512, step=32)
  taux_apprentissage = essai.suggest_float("taux_apprentissage", 1e-4, 1e-2, log=True)

  # Chaque configuration est entraînée plusieurs fois, l'objectif étant la
  # moyenne des justesses de validation obtenues
  justesses_val = []
  for _ in range(NB_EXECUTIONS_PAR_ESSAI):
    modele = nn.Sequential(
      nn.Linear(dim_entree, unites), nn.ReLU(), nn.Linear(unites, nb_classes)
    )
    classifieur = ClassifieurMNIST(
      modele, optimiseur=torch.optim.Adam, taux_apprentissage=taux_apprentissage
    )
    chargeur_entrainement, chargeur_val = fabrique_chargeurs(
      X_train, y_train, taille_lot=1500
    )
    trainer = lightning.Trainer(
      max_epochs=10,
      accelerator="auto",
      devices=1,
      logger=False,
      enable_checkpointing=False,
      enable_progress_bar=False,
      enable_model_summary=False,
    )
    trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
    justesses_val.append(float(trainer.callback_metrics["justesse_val"]))
  return sum(justesses_val) / len(justesses_val)


etude = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
etude.optimize(objectif, n_trials=5)

In [ ]:
print(
  etude.trials_dataframe()[
    ["number", "value", "params_unites", "params_taux_apprentissage"]
  ]
)
print(f"Meilleurs hyper-paramètres : {etude.best_params}")
print(f"Meilleure justesse de validation : {etude.best_value:.4f}")

Le learning rate compte plus que le nombre de neurones. Dans une exécution, le meilleur essai (288 neurones, learning rate de 9,4·10⁻⁴) atteint 95,3 % de justesse de validation, quand les essais au learning rate proche de 10⁻⁴ restent entre 88,5 % et 90,5 %, quelle que soit leur taille : en 10 epochs, un pas si petit n'a pas le temps de converger.

Avec 5 essais, TPE ne guide pas encore la recherche : ses 10 premiers essais sont tirés au hasard (le paramètre `n_startup_trials` de `optuna.samplers.TPESampler`), le temps de réunir assez de résultats pour modéliser le score. Une recherche guidée ne vaut donc que pour des dizaines d'essais.

## AdamW contre SGD sous TensorBoard

TensorBoard affiche les courbes d'apprentissage qu'un [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html), passé au `Trainer` par son argument `logger`, écrit dans un dossier : un sous-dossier par entraînement. Lancez la cellule suivante avant l'exercice : le tableau de bord se remplit au fil des epochs (bouton de rechargement en haut à droite).

Entraînez ensuite deux perceptrons multicouches neufs (`construit_modele_profond()`) avec les réglages de la partie « Perceptron multicouche régularisé », l'un avec `torch.optim.AdamW` et un learning rate de 10⁻³, l'autre avec `torch.optim.SGD` et un learning rate de 0,01, tous deux avec une *weight decay* de 0,01. Passez à chaque `Trainer` un `TensorBoardLogger` qui écrit dans `journaux/adamw` pour le premier, dans `journaux/sgd` pour le second.

Comparez les courbes `perte_val` et `justesse_val` des deux entraînements : lequel converge le plus vite ? Pourquoi ?

In [ ]:
%load_ext tensorboard
%tensorboard --logdir journaux

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `TensorBoardLogger("journaux", name="adamw")` : les courbes vont dans `journaux/adamw`
- Un modèle neuf par optimiseur, `construit_modele_profond()` : sinon le second entraînement repart des poids du premier
- `chargeur_entrainement` et `chargeur_val`, les `DataLoader` de la partie « Perceptron multicouche régularisé », servent aux deux entraînements
- Dans TensorBoard, l'onglet *Scalars* superpose les courbes des deux dossiers ; `CSVLogger` écrit aussi dans `journaux`, mais TensorBoard ignore ses fichiers
- Relancer un entraînement crée un nouveau sous-dossier `version_<n>` : supprimez `journaux` pour repartir d'un tableau de bord propre

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
optimiseurs = {"adamw": (torch.optim.AdamW, 1e-3), "sgd": (torch.optim.SGD, 0.01)}
for nom, (optimiseur, taux_apprentissage) in optimiseurs.items():
  classifieur = ClassifieurMNIST(
    construit_modele_profond(),
    optimiseur=...,
    taux_apprentissage=...,
    decroissance_poids=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("journaux", name=...),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
```

### Solution

In [ ]:
optimiseurs = {"adamw": (torch.optim.AdamW, 1e-3), "sgd": (torch.optim.SGD, 0.01)}
for nom, (optimiseur, taux_apprentissage) in optimiseurs.items():
  classifieur = ClassifieurMNIST(
    construit_modele_profond(),
    optimiseur=optimiseur,
    taux_apprentissage=taux_apprentissage,
    decroissance_poids=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("journaux", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
  print(f"{nom} : justesse_val {float(trainer.callback_metrics['justesse_val']):.3f}")

# AdamW converge bien plus vite : 91 % de justesse de validation dès la première
# epoch et 95 % à la dixième, quand SGD part de 42 % et finit vers 90 %. SGD multiplie
# le gradient de chaque poids par le même learning rate (0,01) ; Adam l'adapte à chaque
# poids, d'après la taille de ses gradients récents, et garde une inertie d'un pas à
# l'autre (la moyenne des gradients).